# 실습 [21-1] 카카오의 딥러닝 기반 형태소 분석기 khaiii(Kakao Hangul Analyzer)

- **관련 장(챕터)**: 21.4 딥러닝 기반 형태소 분석, 품사 태깅 소개
- **실습 목적**: 한국어 형태소 분석에서는 현재 KoNLPy가 가장 보편적으로 쓰이지만, 딥러닝 기반 형태소 분석기도 다양하게 개발되고 있다. 그중 오픈소스로 공개된 카카오의 딥러닝 기반 형태소 분석기 **khaiii**를 직접 빌드하고 사용해 본다.
- **khaiii 특징**: CNN으로 만든 형태소 분석기로 RNN, LSTM 알고리즘보다 속도가 빠르다. 국립국어원 21세기 세종계획 최종 성과물을 전처리한 후 학습한 모델이다. (https://github.com/kakao/khaiii)

> ⚠️ 교재에는 실습 코드가 없어 수업용으로 새로 작성한 노트북이다(교재 원문 아님).
> **Google Colab(CPU 런타임)** 기준이며, khaiii는 pip 패키지가 없어 소스를 직접 빌드한다. **빌드에 15~25분 정도** 걸리므로 수업 전에 1~3단계를 미리 실행해 두는 것을 권장한다.

### 실습 순서
1. 빌드 도구 준비 → 2. 소스 내려받기 → 3. C++ 라이브러리·리소스 빌드 → 4. Python 모듈 설치  
5. 형태소 분석 실행 → 6. 결과 구조 살펴보기 → 7. KoNLPy 분석기와 비교(결과·속도) → 8. 생각해 보기

## 1. 빌드 도구 준비
khaiii는 C++로 작성되어 있어 **CMake**(C++ 빌드 설정 도구)로 빌드한다. 이때 khaiii에 포함된 **Hunter**가 함께 동작하는데, Hunter는 사람이 아니라 **C++용 패키지 관리 도구**다. 파이썬의 `pip`처럼, khaiii가 필요로 하는 외부 라이브러리(Boost, Eigen, fmt 등)를 GitHub에서 자동으로 내려받아 빌드해 준다.
최신 CMake 4.x에서는 khaiii가 사용하는 구버전 Hunter 설정이 거부되므로 **CMake 3.x로 고정**한다.

In [ ]:
!pip install -q "cmake<4"
import os
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"   # CMake 4.x가 남아 있을 때를 대비한 안전장치
!cmake --version | head -1
!g++ --version | head -1

## 2. khaiii 소스 내려받기
⚠️ Python 모듈 설치 스크립트가 내부에서 `cmake ..`를 실행하므로, **빌드 디렉터리는 반드시 소스 폴더 아래(`khaiii/build`)**에 만든다.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/kakao/khaiii.git
%cd /content/khaiii
!mkdir -p build

## 3. C++ 라이브러리와 리소스(사전·모델) 빌드
- `cmake ..` : 빌드 설정을 만든다. 이때 패키지 관리 도구 Hunter가 필요한 외부 라이브러리를 `~/.hunter` 폴더에 내려받아 빌드한다(처음 한 번만 오래 걸리고, 이후에는 저장된 것을 재사용).
- `make all` : khaiii 라이브러리(`lib/libkhaiii.so`)와 실행 파일 빌드
- `make resource` : 형태소 분석에 필요한 리소스(사전·모델 파라미터)를 `share/khaiii`에 생성

In [ ]:
%cd /content/khaiii/build
!cmake .. > cmake.log 2>&1; tail -5 cmake.log

In [ ]:
!make all -j$(nproc) > make_all.log 2>&1; tail -5 make_all.log

In [ ]:
!make resource > make_rsc.log 2>&1; tail -5 make_rsc.log
!ls -lh lib share/khaiii | head -20

## 4. Python 모듈 설치
`make package_python`으로 설치용 패키지를 만들고 pip로 설치한다.
⚠️ 설치가 끝나면 **`/content`로 이동한 뒤** import해야 한다(빌드 폴더 안의 `khaiii.py`가 먼저 import되어 라이브러리를 못 찾는 문제 방지).

In [ ]:
!make package_python > pkg.log 2>&1; tail -3 pkg.log
!pip install -q ./package_python
%cd /content

## 5. 형태소 분석 실행
`KhaiiiApi().analyze(문장)`은 **어절(KhaiiiWord) 목록**을 돌려주고, 각 어절은 **형태소(KhaiiiMorph) 목록**을 가진다.
pip 설치가 실패한 경우를 대비해, 빌드된 라이브러리를 직접 지정해 여는 대체 경로도 넣어 두었다.

In [ ]:
try:
    from khaiii import KhaiiiApi
    api = KhaiiiApi()
except Exception as e:                        # pip 설치 실패 시: 빌드 결과물을 직접 사용
    print("pip 설치 모듈 사용 실패 → 빌드 결과물로 직접 로드:", e)
    import sys
    sys.path.insert(0, "/content/khaiii/src/main/python")
    from khaiii.khaiii import KhaiiiApi
    api = KhaiiiApi("/content/khaiii/build/lib/libkhaiii.so",
                    "/content/khaiii/build/share/khaiii")
print("khaiii version:", api.version())

In [ ]:
for word in api.analyze("안녕, 세상."):
    print(word)            # 어절 \t 형태소/품사 + 형태소/품사 ...

## 6. 결과 구조 살펴보기
- `word.lex` : 어절 원문, `word.begin`·`word.length` : 원문에서의 위치
- `word.morphs` : 형태소 목록 → `morph.lex`(형태소), `morph.tag`(세종 품사 태그)

강의에서 본 예문 **"나는 밥을 먹었다."**와 띄어쓰기가 없는 **"아버지가방에들어가신다"**를 분석해 보자.

In [ ]:
def show(sent):
    print(f"■ {sent}")
    for word in api.analyze(sent):
        morphs = " + ".join(f"{m.lex}/{m.tag}" for m in word.morphs)
        print(f"  {word.lex:<12} → {morphs}")
    print()

for s in ["나는 밥을 먹었다.", "아버지가방에들어가신다", "아버지가 방에 들어가신다"]:
    show(s)

In [ ]:
# 형태소-품사 쌍 리스트로 변환하는 함수 (KoNLPy의 pos()와 같은 형식)
def khaiii_pos(sent):
    return [(m.lex, m.tag) for w in api.analyze(sent) for m in w.morphs]

khaiii_pos("잡히셨겠더라")

**자주 보이는 세종 품사 태그** (강의자료 표 21-1 참고)

| 태그 | 의미 | 태그 | 의미 |
|---|---|---|---|
| NNG | 일반 명사 | JKS | 주격 조사 |
| NNP | 고유 명사 | JKO | 목적격 조사 |
| NP | 대명사 | JKB | 부사격 조사 |
| VV | 동사 | JX | 보조사 |
| VA | 형용사 | EP | 선어말 어미 |
| VX | 보조 용언 | EF / EC | 종결 어미 / 연결 어미 |
| MAG | 일반 부사 | SF / SP | 마침표·물음표 / 쉼표 |

## 7. KoNLPy 형태소 분석기와 비교
같은 문장을 KoNLPy의 Okt, Kkma, Komoran으로 분석해 **분리 결과·품사 체계**와 **속도**를 비교한다.
(KoNLPy는 Java 기반이며 Colab에는 Java가 기본 설치되어 있다.)

In [ ]:
!pip install -q konlpy
from konlpy.tag import Okt, Kkma, Komoran
taggers = {"Okt": Okt(), "Kkma": Kkma(), "Komoran": Komoran()}

sent = "아버지가방에들어가신다"
print(f"{'khaiii':<8}", khaiii_pos(sent))
for name, t in taggers.items():
    print(f"{name:<8}", t.pos(sent))

In [ ]:
import time
sents = ["나는 밥을 먹었다.", "아버지가방에들어가신다",
         "딥러닝 기반 형태소 분석기는 띄어쓰기 오류에도 강하다.",
         "카카오에서 CNN을 사용하여 만든 형태소 분석기로 속도가 빠르다."] * 25   # 100문장

def bench(fn):
    fn(sents[0])                      # 첫 호출(초기화) 제외
    t0 = time.perf_counter()
    for s in sents:
        fn(s)
    return (time.perf_counter() - t0) * 1000

print(f"{'khaiii':<8}: {bench(khaiii_pos):8.1f} ms / 100문장")
for name, t in taggers.items():
    print(f"{name:<8}: {bench(t.pos):8.1f} ms / 100문장")

## 8. 생각해 보기
1. 띄어쓰기가 없는 문장("아버지가방에들어가신다")에서 khaiii와 KoNLPy 분석기들의 결과는 어떻게 다른가? 음절 단위로 입력을 받는 딥러닝 모델(21.4)의 장점과 연결해 설명해 보자.
2. 속도 측정 결과를 비교해 보자. khaiii가 CNN을 사용한 이유(21.4.5)와 연결해 해석해 보자.
3. 신조어·고유명사가 들어간 문장(예: "유한대학교 인공지능전공 학생들이 챗GPT로 과제를 했다")을 분석해 보고, 분석기별로 어떤 오류가 생기는지 찾아보자.

---
### 빌드가 실패할 때 확인할 것
- `cmake.log`에 **`Compatibility with CMake < 3.5 has been removed`** → 1단계의 `cmake<4` 설치가 적용됐는지 `cmake --version`으로 확인 후 런타임을 다시 시작한다.
- `cmake.log`에 **다운로드 실패(download failed)** → 일시적인 네트워크 문제일 수 있으므로 `!rm -rf ~/.hunter build && mkdir build` 후 3단계를 다시 실행한다.
- `import khaiii`에서 **`fail to find library`** → `/content`로 이동했는지 확인하거나, 5단계의 대체 경로(빌드 결과물 직접 로드)를 사용한다.